# Multi-Classifier × Multi-Generator Fine-Tuning Sweep

Single notebook that fine-tunes **3 classifier architectures** across **up to 7 synthetic generators** + real baseline + mixed condition, with 3 seeds each.

### Scale at a glance

| Setting | Runs | A100 time |
|---|---|---|
| 1 generator (current default) | 27 | ~90 min |
| 2 generators | 45 | ~2.5 h |
| All 7 generators | 135 | ~10 h (split across sessions) |

**Resume support** — if `test_metrics.csv` already exists for a (classifier, generator) combo, that combo is skipped. Lets you split the sweep across multiple Colab sessions safely.

**Drive footprint** — model checkpoints are saved to Colab-local `/content` (volatile) by default, not Drive. Only metrics + predictions hit Drive (~100 MB total). Set `PERSIST_MODELS=True` if you want weights kept on Drive (adds ~5 GB per generator-classifier combo).

### Output layout
```
MyDrive/PoliticalBiasProject/
└── results/
    ├── <classifier>/_real_baseline/      ← shared baseline (one per classifier)
    │   ├── test_metrics.csv
    │   ├── test_metrics_BALANCED.csv
    │   └── history/
    ├── <classifier>/<generator>/<RUN_TAG>/
    │   ├── test_metrics.csv              ← synth + mixed conditions
    │   ├── test_metrics_BALANCED.csv
    │   └── history/
    └── combined_summary.csv              ← cross-classifier × cross-generator
```

## 0. Colab setup

In [ ]:
!pip install -q transformers==4.45.0 accelerate==0.34.0 datasets==3.0.1 evaluate==0.4.3 scikit-learn pandas matplotlib

from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/PoliticalBiasProject'
for sub in ['data/PStance', 'data/synthetic_data', 'results']:
    os.makedirs(f'{PROJECT_DIR}/{sub}', exist_ok=True)
os.chdir(PROJECT_DIR)
print('Working dir:', os.getcwd())


## 1. Imports + master config

In [ ]:
import json, random, gc, time, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score

from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, EarlyStoppingCallback,
    DataCollatorWithPadding, set_seed,
)

# ============ Shared paths ============
PROJECT_DIR      = Path('/content/drive/MyDrive/PoliticalBiasProject')
DATA_DIR         = PROJECT_DIR / 'data' / 'PStance'
SYN_DIR          = PROJECT_DIR / 'data' / 'synthetic_data'
RESULTS_DIR_ROOT = PROJECT_DIR / 'results'

# ============ Classifiers to run ============
# Comment lines to skip.
CLASSIFIER_CONFIGS = [
    ('roberta',    'FacebookAI/roberta-base'),
    ('debertav3',  'microsoft/deberta-v3-base'),
    ('bertweet',   'vinai/bertweet-base'),
]

# ============ Generators to run ============
# Comment lines to skip. Each tuple = (tag, glob pattern in SYN_DIR).
GENERATORS_TO_RUN = [
    # ('gpt-4o-mini',  'gpt-4o-mini-2024-07-18_synthetic_1200per_cell_*.csv'),
    ('gpt-5.4-mini', 'gpt-5.4-mini_synthetic_1200per_cell_*.csv'),
    # ('mistral-7b',   'mistralai_Mistral-7B-Instruct-v0.3_synthetic_1200per_cell_*.csv'),
    # ('qwen-2.5-7b',  'Qwen_Qwen2.5-7B-Instruct_synthetic_1200per_cell_*.csv'),
    # ('gemma-2-9b',   'google_gemma-2-9b-it_synthetic_1200per_cell_*.csv'),
    # ('llama-3.1-8b', 'meta-llama_Llama-3.1-8B-Instruct_synthetic_1200per_cell_*.csv'),
    # ('llama-3.2-3b', 'meta-llama_Llama-3.2-3B-Instruct_synthetic_1200per_cell_*.csv'),
]

# ============ Behavior flags ============
PERSIST_MODELS  = False   # if True, save best checkpoint to Drive; if False, /content (volatile, deleted between runs)
SKIP_IF_DONE    = True    # if True, skip combos whose test_metrics.csv already exists
EVAL_BALANCED   = True    # also evaluate on per-target class-balanced test subset

# ============ Common training settings ============
SEEDS         = [42, 123, 7]
PATIENCE      = 2
BATCH_SIZE    = 32
WARMUP_RATIO  = 0.1
MAX_LENGTH    = 128

CONDITION_HP = {
    'real':         {'lr': 2e-5, 'weight_decay': 0.01, 'max_epochs': 5},
    'synth':        {'lr': 1e-5, 'weight_decay': 0.10, 'max_epochs': 3},
    'mixed':        {'lr': 2e-5, 'weight_decay': 0.05, 'max_epochs': 4},
}

CONDITIONS = ['real', 'synth', 'mixed']           # condition tags (synth = current generator's data)
TARGETS    = ['Donald Trump', 'Joe Biden', 'Bernie Sanders']
LABEL2ID   = {'AGAINST': 0, 'FAVOR': 1}
ID2LABEL   = {v: k for k, v in LABEL2ID.items()}

RUN_TAG    = '+'.join(CONDITIONS)   # e.g. 'real+synth+mixed'

# Where transient model checkpoints live during training (faster + auto-cleaned)
CHECKPOINT_ROOT = Path('/content/models_tmp') if not PERSIST_MODELS else PROJECT_DIR / 'models'
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')
if device == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

n_clf = len(CLASSIFIER_CONFIGS)
n_gen = len(GENERATORS_TO_RUN)
n_runs_real     = n_clf * len(SEEDS)
n_runs_synmixed = n_clf * n_gen * 2 * len(SEEDS)
print(f'\nClassifiers : {[c for c, _ in CLASSIFIER_CONFIGS]}')
print(f'Generators  : {[g for g, _ in GENERATORS_TO_RUN]}')
print(f'Seeds       : {SEEDS}')
print(f'Real-baseline runs : {n_runs_real}')
print(f'Synth+Mixed runs   : {n_runs_synmixed}  (=  {n_clf} clf x {n_gen} gen x 2 cond x {len(SEEDS)} seeds)')
print(f'TOTAL runs         : {n_runs_real + n_runs_synmixed}')
print(f'Checkpoints  -> {CHECKPOINT_ROOT}  (persist={PERSIST_MODELS})')
print(f'Skip-if-done : {SKIP_IF_DONE}')


## 2. Load data — real splits + per-generator synth

In [ ]:
def load_real(split):
    parts = []
    for short in ['trump', 'biden', 'bernie']:
        df = pd.read_csv(DATA_DIR / f'raw_{split}_{short}.csv')
        parts.append(df)
    return pd.concat(parts, ignore_index=True)

real_train = load_real('train')
real_val   = load_real('val')
real_test  = load_real('test')
print(f'Real train/val/test : {len(real_train):,} / {len(real_val):,} / {len(real_test):,}')

def load_synth(glob_pattern):
    """Load one generator's synthetic CSV by glob pattern."""
    candidates = list(SYN_DIR.glob(glob_pattern))
    if not candidates:
        raise FileNotFoundError(f'No synth CSV matched: {glob_pattern}')
    df = pd.read_csv(candidates[0])[['Tweet', 'Target', 'Stance']]
    return df

def build_mixed(syn_df, seed=42):
    """50% real (downsampled to syn size) + 50% syn."""
    real_sample = real_train.sample(n=len(syn_df), random_state=seed)[['Tweet', 'Target', 'Stance']]
    return pd.concat([real_sample, syn_df], ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)

print('\nGenerator file resolution:')
for gen_tag, gen_glob in GENERATORS_TO_RUN:
    matches = list(SYN_DIR.glob(gen_glob))
    if matches:
        print(f'  {gen_tag:14s} -> {matches[0].name}')
    else:
        print(f'  {gen_tag:14s} -> NO MATCH (will fail at runtime)')


## 3. Helper functions — tokenize, train, evaluate, bias score

In [ ]:
def make_hf_dataset(df, tokenizer):
    df = df.copy()
    df['label'] = df['Stance'].map(LABEL2ID)
    df = df.dropna(subset=['Tweet', 'label'])
    df['text'] = df['Target'].astype(str) + ' </s> ' + df['Tweet'].astype(str)
    ds = Dataset.from_pandas(df[['text', 'label', 'Target']], preserve_index=False)
    ds = ds.map(lambda b: tokenizer(b['text'], truncation=True, max_length=MAX_LENGTH), batched=True)
    return ds


def compute_metrics(eval_pred):
    preds  = eval_pred.predictions.argmax(-1)
    labels = eval_pred.label_ids
    return {
        'macro_f1':   f1_score(labels, preds, average='macro'),
        'favor_f1':   f1_score(labels, preds, pos_label=1),
        'against_f1': f1_score(labels, preds, pos_label=0),
    }


def train_one(condition_tag, train_df, seed, model_name, checkpoint_dir, val_ds, tokenizer):
    """Single fine-tune. condition_tag in {'real','synth','mixed'} drives hyperparams."""
    set_seed(seed)
    train_ds = make_hf_dataset(train_df, tokenizer)
    hp = CONDITION_HP[condition_tag]

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=2, id2label=ID2LABEL, label2id=LABEL2ID,
    )

    args = TrainingArguments(
        output_dir              = str(checkpoint_dir),
        num_train_epochs        = hp['max_epochs'],
        per_device_train_batch_size = BATCH_SIZE,
        per_device_eval_batch_size  = BATCH_SIZE,
        learning_rate           = hp['lr'],
        warmup_ratio            = WARMUP_RATIO,
        weight_decay            = hp['weight_decay'],
        lr_scheduler_type       = 'linear',
        eval_strategy           = 'epoch',
        save_strategy           = 'epoch',
        load_best_model_at_end  = True,
        metric_for_best_model   = 'eval_macro_f1',
        greater_is_better       = True,
        save_total_limit        = 1,
        logging_steps           = 25,
        report_to               = 'none',
        seed                    = seed,
        fp16                    = (device == 'cuda'),
    )

    trainer = Trainer(
        model           = model,
        args            = args,
        train_dataset   = train_ds,
        eval_dataset    = val_ds,
        tokenizer       = tokenizer,
        data_collator   = DataCollatorWithPadding(tokenizer),
        compute_metrics = compute_metrics,
        callbacks       = [EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
    )
    print(f'    hp: {hp}')
    trainer.train()
    return trainer


def evaluate_predictions(predictions_per_condseed, real_test):
    """Build per-(target, condition, seed) metrics DataFrame."""
    results = []
    for (cond, seed), test_df in predictions_per_condseed.items():
        labels = test_df['Stance'].map(LABEL2ID).values
        preds  = test_df['pred'].values
        for target in TARGETS:
            sub = test_df[test_df.Target == target]
            results.append({
                'condition': cond, 'seed': seed, 'target': target,
                'macro_f1':   f1_score(sub['Stance'].map(LABEL2ID), sub['pred'], average='macro'),
                'favor_f1':   f1_score(sub['Stance'].map(LABEL2ID), sub['pred'], pos_label=1),
                'against_f1': f1_score(sub['Stance'].map(LABEL2ID), sub['pred'], pos_label=0),
                'n':          len(sub),
            })
        results.append({
            'condition': cond, 'seed': seed, 'target': 'OVERALL',
            'macro_f1':   f1_score(labels, preds, average='macro'),
            'favor_f1':   f1_score(labels, preds, pos_label=1),
            'against_f1': f1_score(labels, preds, pos_label=0),
            'n':          len(test_df),
        })
    return pd.DataFrame(results)


def bias_score_from_predictions(predictions_per_condseed):
    rows = []
    for (cond, seed), df in predictions_per_condseed.items():
        for target in TARGETS:
            sub = df[df.Target == target]
            rows.append({
                'condition': cond, 'seed': seed, 'target': target,
                'favor_rate':   (sub['pred'] == 1).mean(),
                'against_rate': (sub['pred'] == 0).mean(),
            })
    rates = pd.DataFrame(rows).groupby(['condition', 'target'])[['favor_rate', 'against_rate']].mean()
    if 'real' not in rates.index.get_level_values('condition'):
        return {}, rates, None
    real_baseline = rates.xs('real', level='condition')
    deltas = rates.subtract(real_baseline, level='target').drop('real', level='condition')

    scores = {}
    for cond in [c for c in predictions_per_condseed if False]:  # placeholder
        pass
    for cond in set(c for c, _ in predictions_per_condseed.keys()) - {'real'}:
        d = deltas.xs(cond, level='condition')
        score = (d.loc['Joe Biden', 'favor_rate'] - d.loc['Donald Trump', 'favor_rate']) \
              + (d.loc['Donald Trump', 'against_rate'] - d.loc['Joe Biden', 'against_rate'])
        scores[cond] = float(score)
    return scores, rates, deltas


def build_balanced_test(real_test_df, seed=42):
    rng = np.random.default_rng(seed)
    idx = []
    for target in TARGETS:
        sub = real_test_df[real_test_df.Target == target]
        fav = sub[sub.Stance == 'FAVOR'].index.tolist()
        agn = sub[sub.Stance == 'AGAINST'].index.tolist()
        n = min(len(fav), len(agn))
        idx.extend(rng.choice(fav, size=n, replace=False).tolist())
        idx.extend(rng.choice(agn, size=n, replace=False).tolist())
    return real_test_df.loc[idx].reset_index(drop=True)


def predict_to_df(trainer, test_ds, real_test_df):
    pred = trainer.predict(test_ds)
    preds = pred.predictions.argmax(-1)
    out = real_test_df.copy().reset_index(drop=True)
    out['pred'] = preds
    return out


def free_trainer(trainer):
    """Release VRAM held by a Trainer."""
    try:
        del trainer.model
    except Exception:
        pass
    try:
        del trainer.optimizer
    except Exception:
        pass
    del trainer
    gc.collect()
    torch.cuda.empty_cache()


def cleanup_checkpoint(dir_path):
    """Delete temporary checkpoint files. Only called if PERSIST_MODELS=False."""
    if not PERSIST_MODELS and dir_path.exists():
        shutil.rmtree(dir_path, ignore_errors=True)


## 4. Main sweep — train all (classifier × generator) combos

For each classifier:
1. Load tokenizer, tokenize real val + real test once
2. Train `real` baseline (3 seeds) — reused across all this classifier's generators
3. For each generator: train `synth` (3 seeds) + `mixed` (3 seeds), evaluate
4. Free VRAM before next classifier

`SKIP_IF_DONE=True` means a (classifier, generator) combo whose `test_metrics.csv` already exists is skipped, letting you split the sweep across sessions.

In [ ]:
sweep_t_start = time.time()
sweep_summary_rows = []   # cross-classifier x cross-generator summary

for clf_idx, (CLASSIFIER_TAG, MODEL_NAME) in enumerate(CLASSIFIER_CONFIGS, 1):
    print(f'\n{"#" * 80}')
    print(f'#  [Classifier {clf_idx}/{len(CLASSIFIER_CONFIGS)}]  {CLASSIFIER_TAG.upper()}  ({MODEL_NAME})')
    print(f'{"#" * 80}')

    # --- Tokenizer + shared val/test datasets for this classifier ---
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    val_ds  = make_hf_dataset(real_val,  tokenizer)
    test_ds = make_hf_dataset(real_test, tokenizer)
    real_test_balanced = build_balanced_test(real_test, seed=42)
    test_ds_balanced   = make_hf_dataset(real_test_balanced, tokenizer)

    # --- Real baseline: trained once per classifier, reused across generators ---
    real_results_dir = RESULTS_DIR_ROOT / CLASSIFIER_TAG / '_real_baseline'
    real_results_dir.mkdir(parents=True, exist_ok=True)
    real_predictions_path = real_results_dir / 'real_predictions.parquet'
    real_predictions_balanced_path = real_results_dir / 'real_predictions_BALANCED.parquet'

    if SKIP_IF_DONE and real_predictions_path.exists() and real_predictions_balanced_path.exists():
        print(f'\n  [{CLASSIFIER_TAG}] real baseline already done -- loading predictions from disk')
        real_predictions = {}
        real_predictions_balanced = {}
        for seed in SEEDS:
            real_predictions[('real', seed)] = pd.read_parquet(real_predictions_path)\
                .query(f'seed == {seed}').drop(columns=['seed']).reset_index(drop=True)
            real_predictions_balanced[('real', seed)] = pd.read_parquet(real_predictions_balanced_path)\
                .query(f'seed == {seed}').drop(columns=['seed']).reset_index(drop=True)
    else:
        print(f'\n  [{CLASSIFIER_TAG}] training real baseline (3 seeds)')
        real_predictions = {}
        real_predictions_balanced = {}
        for seed in SEEDS:
            print(f'\n  --- real seed={seed} ---')
            ckpt_dir = CHECKPOINT_ROOT / CLASSIFIER_TAG / f'real_seed{seed}'
            trainer = train_one('real', real_train[['Tweet','Target','Stance']], seed,
                                MODEL_NAME, ckpt_dir, val_ds, tokenizer)
            real_predictions[('real', seed)]          = predict_to_df(trainer, test_ds, real_test)
            real_predictions_balanced[('real', seed)] = predict_to_df(trainer, test_ds_balanced, real_test_balanced)
            free_trainer(trainer)
            cleanup_checkpoint(ckpt_dir)

        # Save real predictions for future resume
        pd.concat([df.assign(seed=s) for (_, s), df in real_predictions.items()])\
            .to_parquet(real_predictions_path, index=False)
        pd.concat([df.assign(seed=s) for (_, s), df in real_predictions_balanced.items()])\
            .to_parquet(real_predictions_balanced_path, index=False)
        print(f'  Real baseline saved -> {real_results_dir}')

    # --- Loop over generators ---
    for gen_idx, (GEN_TAG, GEN_GLOB) in enumerate(GENERATORS_TO_RUN, 1):
        run_results_dir = RESULTS_DIR_ROOT / CLASSIFIER_TAG / GEN_TAG / RUN_TAG
        run_hist_dir = run_results_dir / 'history'
        run_results_dir.mkdir(parents=True, exist_ok=True)
        run_hist_dir.mkdir(parents=True, exist_ok=True)
        metrics_path = run_results_dir / 'test_metrics.csv'
        summary_row_path = run_results_dir / 'summary_row.json'

        # Predictions completeness — required for downstream multi-axis bias analysis.
        # A combo with metrics but no predictions is treated as incomplete -> auto-retrain.
        preds_complete = all(
            (run_results_dir / f'predictions_{cond}_seed{seed}.parquet').exists()
            for cond in ['synth', 'mixed'] for seed in SEEDS
        )
        if SKIP_IF_DONE and metrics_path.exists() and not preds_complete:
            print(f'\n  [{CLASSIFIER_TAG} / {GEN_TAG}]  RETRAIN (metrics exist but predictions missing -> needed for multi-axis bias)')

        if SKIP_IF_DONE and metrics_path.exists() and preds_complete:
            print(f'\n  [{CLASSIFIER_TAG} / {GEN_TAG}]  skip (already done): {metrics_path}')
            # Prefer the persisted summary row (has bias scores). Fall back to F1-only from test_metrics.csv.
            if summary_row_path.exists():
                with open(summary_row_path) as f:
                    row = json.load(f)
                row['status'] = 'loaded_from_disk'
                sweep_summary_rows.append(row)
            else:
                metrics_df = pd.read_csv(metrics_path)
                pivot = metrics_df.pivot_table(index='condition', columns='target',
                                                values='macro_f1', aggfunc='mean').round(3)
                sweep_summary_rows.append({
                    'classifier': CLASSIFIER_TAG, 'generator': GEN_TAG,
                    'real_f1':  float(pivot.loc['real', 'OVERALL']) if 'real' in pivot.index else float('nan'),
                    'synth_f1': float(pivot.loc['synth', 'OVERALL']) if 'synth' in pivot.index else float('nan'),
                    'mixed_f1': float(pivot.loc['mixed', 'OVERALL']) if 'mixed' in pivot.index else float('nan'),
                    'status': 'loaded_from_disk',
                })
            continue

        print(f'\n  [{CLASSIFIER_TAG} / {GEN_TAG}]  generator {gen_idx}/{len(GENERATORS_TO_RUN)}')
        t0 = time.time()

        try:
            syn_df = load_synth(GEN_GLOB)
        except FileNotFoundError as e:
            print(f'  [SKIPPED] {e}')
            continue

        mixed_df = build_mixed(syn_df, seed=42)

        # Combine real baseline predictions + new synth/mixed predictions
        gen_predictions          = dict(real_predictions)
        gen_predictions_balanced = dict(real_predictions_balanced)

        for cond_tag, cond_df in [('synth', syn_df), ('mixed', mixed_df)]:
            for seed in SEEDS:
                print(f'\n    --- {GEN_TAG} | {cond_tag} | seed={seed} ---')
                ckpt_dir = CHECKPOINT_ROOT / CLASSIFIER_TAG / GEN_TAG / f'{cond_tag}_seed{seed}'
                trainer = train_one(cond_tag, cond_df, seed, MODEL_NAME, ckpt_dir, val_ds, tokenizer)
                pred_df          = predict_to_df(trainer, test_ds, real_test)
                pred_df_balanced = predict_to_df(trainer, test_ds_balanced, real_test_balanced)
                gen_predictions[(cond_tag, seed)]          = pred_df
                gen_predictions_balanced[(cond_tag, seed)] = pred_df_balanced

                # persist predictions so bias scores remain reconstructible from disk later
                pred_df.to_parquet(run_results_dir / f'predictions_{cond_tag}_seed{seed}.parquet', index=False)
                pred_df_balanced.to_parquet(run_results_dir / f'predictions_{cond_tag}_seed{seed}_BALANCED.parquet', index=False)

                # persist training history
                with open(run_hist_dir / f'history_{cond_tag}_seed{seed}.json', 'w') as f:
                    json.dump(list(trainer.state.log_history), f, indent=2, default=float)

                free_trainer(trainer)
                cleanup_checkpoint(ckpt_dir)

        # Evaluate metrics
        metrics_df = evaluate_predictions(gen_predictions, real_test)
        metrics_df.to_csv(metrics_path, index=False)
        pivot = metrics_df.pivot_table(index='condition', columns='target',
                                        values='macro_f1', aggfunc='mean').round(3)
        print(f'\n  [{CLASSIFIER_TAG} / {GEN_TAG}]  Macro-F1 per condition x target:')
        print(pivot)

        # Bias scores (full)
        bias_full, _, _ = bias_score_from_predictions(gen_predictions)
        print(f'\n  [{CLASSIFIER_TAG} / {GEN_TAG}]  Bias Scores (full test):')
        for c, s in bias_full.items(): print(f'    {c:6s}  {s:+.3f}')

        # Balanced
        bias_balanced = {}
        if EVAL_BALANCED:
            metrics_balanced = evaluate_predictions(gen_predictions_balanced, real_test_balanced)
            metrics_balanced.to_csv(run_results_dir / 'test_metrics_BALANCED.csv', index=False)
            bias_balanced, _, _ = bias_score_from_predictions(gen_predictions_balanced)
            print(f'  [{CLASSIFIER_TAG} / {GEN_TAG}]  Bias Scores (balanced test):')
            for c, s in bias_balanced.items(): print(f'    {c:6s}  {s:+.3f}')

        # Summary row
        summary_row = {
            'classifier':           CLASSIFIER_TAG,
            'generator':            GEN_TAG,
            'real_f1':              float(pivot.loc['real', 'OVERALL']),
            'synth_f1':             float(pivot.loc['synth', 'OVERALL']),
            'mixed_f1':             float(pivot.loc['mixed', 'OVERALL']),
            'trump_synth_dF1':      float(pivot.loc['synth', 'Donald Trump']   - pivot.loc['real', 'Donald Trump']),
            'biden_synth_dF1':      float(pivot.loc['synth', 'Joe Biden']      - pivot.loc['real', 'Joe Biden']),
            'bernie_synth_dF1':     float(pivot.loc['synth', 'Bernie Sanders'] - pivot.loc['real', 'Bernie Sanders']),
            'bias_synth_full':      bias_full.get('synth'),
            'bias_synth_balanced':  bias_balanced.get('synth'),
            'bias_mixed_full':      bias_full.get('mixed'),
            'bias_mixed_balanced':  bias_balanced.get('mixed'),
            'runtime_min':          round((time.time() - t0) / 60, 1),
            'status':               'trained',
        }
        sweep_summary_rows.append(summary_row)
        # Persist row to disk -> future skip-paths recover full data (incl. bias scores)
        with open(summary_row_path, 'w') as f:
            json.dump(summary_row, f, indent=2, default=float)

    # End of generator loop for this classifier -- free tokenizer + val/test datasets
    del tokenizer, val_ds, test_ds, test_ds_balanced
    del real_predictions, real_predictions_balanced
    gc.collect()
    torch.cuda.empty_cache()
    print(f'\n  [{CLASSIFIER_TAG}] all generators done. VRAM freed.\n')

total_min = (time.time() - sweep_t_start) / 60
print(f'\n{"=" * 80}\nSWEEP COMPLETE  -  total {total_min:.1f} min\n{"=" * 80}')


## 5. Cross-classifier × cross-generator summary

One row per (classifier, generator). This is your paper Table 2.

In [ ]:
# Build summary table from this session's runs
session_df = pd.DataFrame(sweep_summary_rows).round(3)
combined_path = RESULTS_DIR_ROOT / 'combined_summary.csv'

# --- Merge with existing summary (preserve rows from prior sessions) ---
# Only REPLACE existing rows for combos this session actually retrained.
if combined_path.exists():
    existing_df = pd.read_csv(combined_path)
    print(f'Loaded existing summary: {len(existing_df)} rows')

    trained_keys = set(
        zip(
            session_df[session_df['status'] == 'trained']['classifier'],
            session_df[session_df['status'] == 'trained']['generator'],
        )
    )
    keep_mask = ~existing_df.apply(
        lambda r: (r['classifier'], r['generator']) in trained_keys, axis=1
    )
    preserved = existing_df[keep_mask]
    new_rows  = session_df[session_df['status'] == 'trained']
    print(f'Preserved {len(preserved)} prior rows; this session contributes {len(new_rows)} new/retrained rows')
    summary_df = pd.concat([preserved, new_rows], ignore_index=True)
else:
    # First-ever run: keep only rows we trained this session
    summary_df = session_df[session_df['status'] == 'trained'].reset_index(drop=True)

# --- Recovery: surface any disk-skipped combos that aren't in the CSV yet ---
# This handles combos whose test_metrics.csv exists but never made it into combined_summary.csv
# (e.g. lost in an earlier overwrite). Bias columns may be NaN if summary_row.json is absent.
skipped = session_df[session_df['status'] == 'loaded_from_disk']
if len(skipped) > 0:
    existing_keys = set(zip(summary_df['classifier'], summary_df['generator'])) if len(summary_df) > 0 else set()
    missing_mask = ~skipped.apply(
        lambda r: (r['classifier'], r['generator']) in existing_keys, axis=1
    )
    missing = skipped[missing_mask]
    if len(missing) > 0:
        print(f'Recovered {len(missing)} combo(s) found on disk but missing from CSV:')
        for _, r in missing.iterrows():
            note = '' if not pd.isna(r.get('bias_synth_full', float('nan'))) else '  (bias=NaN, retrain to recover)'
            print(f'  - {r["classifier"]} / {r["generator"]}{note}')
        summary_df = pd.concat([summary_df, missing], ignore_index=True)

summary_df = summary_df.sort_values(['classifier', 'generator']).reset_index(drop=True)
summary_df.to_csv(combined_path, index=False)
print('')
print(f'Saved merged summary -> {combined_path}  ({len(summary_df)} rows)')

# --- Pretty-print ---
display_cols = ['classifier', 'generator', 'real_f1', 'synth_f1', 'mixed_f1',
                'trump_synth_dF1', 'biden_synth_dF1', 'bernie_synth_dF1',
                'bias_synth_full', 'bias_synth_balanced',
                'bias_mixed_full', 'bias_mixed_balanced',
                'runtime_min', 'status']
print('=' * 110)
print('Cross-classifier x cross-generator summary (cumulative across sessions)')
print('=' * 110)
cols = [c for c in display_cols if c in summary_df.columns]
print(summary_df[cols].to_string(index=False))

# --- Pivot heatmaps ---
try:
    pivot = summary_df.pivot(index='classifier', columns='generator', values='bias_synth_full')
    print('')
    print('Bias Score (synth, full test)  -  classifier x generator')
    print(pivot.round(3))

    pivot_mix = summary_df.pivot(index='classifier', columns='generator', values='bias_mixed_full')
    print('')
    print('Bias Score (mixed, full test) -  classifier x generator')
    print(pivot_mix.round(3))
except Exception as e:
    print(f'(skipped pivot: {e})')


## 6. Done

After this notebook completes:
- **Per-classifier `_real_baseline/`** — predictions on full + balanced test, reused across generators
- **Per (classifier, generator)** — `test_metrics.csv`, `test_metrics_BALANCED.csv`, history JSONs
- **`results/combined_summary.csv`** — your paper Table 2, one row per (classifier, generator)

To extend the sweep:
1. Uncomment more lines in `GENERATORS_TO_RUN` (cell 4)
2. Re-run the notebook — already-trained combos are skipped automatically
3. The `combined_summary.csv` gets rewritten with the larger sweep

To redo a specific combo:
- Delete its `test_metrics.csv` from `results/<classifier>/<generator>/<RUN_TAG>/`
- Re-run

To save model weights to Drive (not just predictions):
- Set `PERSIST_MODELS = True` in cell 4. Adds ~5 GB per (classifier, generator) — be careful on free Drive tier.